In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

In this document we train logistic regression on one complication in a federated setup. In general we observe results close to those of multiple regression.

## Federated Logistic Regression

Logistic regression models the probability of a complication with the sigmoid function

$$
P(Y=1 \mid X=x)
=
\sigma(\mathbf{x}^{\top}\mathbf{w} + b),
$$

where

$$
\sigma(z)
=
\frac{1}{1+e^{-z}},
$$

$\mathbf{w}$ denotes the coefficient vector and $b$ the intercept.

For a dataset with $n$ observations, the parameters are estimated by minimizing the binary cross-entropy loss

$$
L(\mathbf{w},b)
=
-\frac{1}{n}
\sum_{i=1}^{n}
\left[
y_i \log(p_i)
+
(1-y_i)\log(1-p_i)
\right],
$$

where

$$
p_i
=
\sigma(\mathbf{x}_i^{\top}\mathbf{w}+b).
$$

In the federated setting, the training data are distributed across $K$ clients. Let $D_k$ denote the local dataset of client $k$, containing $n_k$ observations. At federated  round $t$, the server sends the current global parameters

$$
(\mathbf{w}^{(t)}, b^{(t)})
$$

to all clients.

Client $k$ initializes its local logistic regression model with the global parameters and performs one SGD update (One gradient descent step per pations) on its own data, producing updated parameters

$$
(\mathbf{w}_k^{(t+1)}, b_k^{(t+1)}).
$$

The server then aggregates the local models using the FedAvg algorithm

$$
\mathbf{w}^{(t+1)}
=
\sum_{k=1}^{K}
\frac{n_k}{N}
\mathbf{w}_k^{(t+1)},
$$

and

$$
b^{(t+1)}
=
\sum_{k=1}^{K}
\frac{n_k}{N}
b_k^{(t+1)},
$$

where

$$
N
=
\sum_{k=1}^{K} n_k.
$$

Thus, clients with more observations contribute proportionally more to the global model.



## Data

In [2]:
path_train = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\raw\synthetic_dataset_A_non-iid.csv"
path_test = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\processed\A\global_test_set_non-iid.csv"

df = pd.read_csv(path_train)
test_data = pd.read_csv(path_test)

"""
df = (
    df.groupby("Client", group_keys=False)
      .apply(lambda x: x.sample(frac=20000 / len(df), random_state=42))
      .reset_index(drop=True)
)
"""


variables = df.columns[2:27].tolist()

# Ændre Target_col for øvrige complicationer.

target_col = "Risk_NerveDysesthesia"
category_col = "Risk_Category_NerveDysesthesia"
client_col = "Client"


scaler = StandardScaler()

train_design = pd.DataFrame(
    scaler.fit_transform(df[variables]),
    columns=variables,
    index=df.index
)

test_design = pd.DataFrame(
    scaler.transform(test_data[variables]),
    columns=variables,
    index=test_data.index
)

train_design[target_col] = df[target_col].values
train_design[category_col] = df[category_col].values
train_design[client_col] = df[client_col].values

test_design[target_col] = test_data[target_col].values
test_design[category_col] = test_data[category_col].values



In [3]:
len(df)

47000

## Functions

In [4]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def assign_tertiles(y_hat):
    q1 = np.quantile(y_hat, 1/3)
    q2 = np.quantile(y_hat, 2/3)

    groups = np.where(
        y_hat <= q1, 0,
        np.where(y_hat <= q2, 1, 2)
    )

    return groups, q1, q2


def metrics_for_one_complication(y_true, y_pred):
    results = {}

    for cls in [0, 1, 2]:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0 else 0.0
        )

        results[cls] = {
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

    macro_f1 = np.mean([results[cls]["f1"] for cls in [0, 1, 2]])

    return {
        "per_class": results,
        "macro_f1": macro_f1
    }


def federated_logistic_regression_binary(
    train_df,
    test_df,
    variables,
    target_col,
    category_col,
    client_col="Client",
    rounds=50,
    local_epochs=5,
    lr=0.01,
    weighted=True,
    random_state=42
):
    n_features = len(variables)

    global_coef = np.zeros((1, n_features))
    global_intercept = np.zeros(1)

    X_test = test_df[variables].to_numpy(dtype=float)
    y_true_group = test_df[category_col].to_numpy(dtype=int)

    history = []

    for rnd in range(rounds):

        old_coef = global_coef.copy()
        old_intercept = global_intercept.copy()

        local_coefs = []
        local_intercepts = []
        local_sizes = []

        for client_id in sorted(train_df[client_col].unique()):

            client_df = train_df[train_df[client_col] == client_id]

            X_client = client_df[variables].to_numpy(dtype=float)
            y_client = client_df[target_col].to_numpy(dtype=int)

            model = SGDClassifier(
                loss="log_loss",
                penalty=None,
                learning_rate="constant",
                eta0=lr,
                max_iter=1,
                tol=None,
                warm_start=True,
                random_state=random_state
            )

            # Dummy initialisering
            model.partial_fit(
                X_client[:1],
                y_client[:1],
                classes=np.array([0, 1])
            )

            # Overskriv med globale vægte
            model.coef_ = global_coef.copy()
            model.intercept_ = global_intercept.copy()
            model.classes_ = np.array([0, 1])

            # Lokal træning
            for _ in range(local_epochs):
                model.partial_fit(X_client, y_client)

            local_coefs.append(model.coef_.copy())
            local_intercepts.append(model.intercept_.copy())
            local_sizes.append(len(client_df))

        local_sizes = np.array(local_sizes, dtype=float)

        if weighted:
            global_coef = np.average(local_coefs, axis=0, weights=local_sizes)
            global_intercept = np.average(local_intercepts, axis=0, weights=local_sizes)
        else:
            global_coef = np.mean(local_coefs, axis=0)
            global_intercept = np.mean(local_intercepts, axis=0)

        coef_change = np.linalg.norm(global_coef - old_coef)
        intercept_change = np.linalg.norm(global_intercept - old_intercept)

        logits = X_test @ global_coef.T + global_intercept
        probs = sigmoid(logits).ravel()

        y_pred_group, q1, q2 = assign_tertiles(probs)

        metrics = metrics_for_one_complication(y_true_group, y_pred_group)
        macro_f1 = metrics["macro_f1"]

        history.append({
            "round": rnd + 1,
            "macro_f1": macro_f1,
            "q1": q1,
            "q2": q2,
            "coef_change": coef_change,
            "intercept_change": intercept_change
        })

        print(
            f"Round {rnd+1:03d} | Macro F1 = {macro_f1:.4f} | "
            f"coef_change = {coef_change:.6f} | "
            f"intercept_change = {intercept_change:.6f}"
        )

    return global_intercept, global_coef, pd.DataFrame(history)

## Traning

In [5]:
intercept, coefs, history_df = federated_logistic_regression_binary(
    train_df=train_design,
    test_df=test_design,
    variables=variables,
    target_col=target_col,
    category_col=category_col,
    client_col=client_col,
    rounds=50,
    local_epochs=1,
    lr=0.001,
    weighted=True
)

Round 001 | Macro F1 = 0.0873 | coef_change = 0.054319 | intercept_change = 1.638424
Round 002 | Macro F1 = 0.1031 | coef_change = 0.015945 | intercept_change = 0.708766
Round 003 | Macro F1 = 0.2245 | coef_change = 0.024165 | intercept_change = 0.422748
Round 004 | Macro F1 = 0.3712 | coef_change = 0.026996 | intercept_change = 0.292749
Round 005 | Macro F1 = 0.4739 | coef_change = 0.028059 | intercept_change = 0.219873
Round 006 | Macro F1 = 0.5404 | coef_change = 0.028399 | intercept_change = 0.173719
Round 007 | Macro F1 = 0.5761 | coef_change = 0.028383 | intercept_change = 0.142085
Round 008 | Macro F1 = 0.6052 | coef_change = 0.028167 | intercept_change = 0.119176
Round 009 | Macro F1 = 0.6223 | coef_change = 0.027830 | intercept_change = 0.101901
Round 010 | Macro F1 = 0.6337 | coef_change = 0.027414 | intercept_change = 0.088465
Round 011 | Macro F1 = 0.6455 | coef_change = 0.026945 | intercept_change = 0.077759
Round 012 | Macro F1 = 0.6558 | coef_change = 0.026439 | intercep

## Results

In [6]:
X_test_final = test_design[variables].to_numpy(dtype=float)
y_true_group = test_design[category_col].to_numpy(dtype=int)

logits = X_test_final @ coefs.T + intercept
probs = sigmoid(logits).ravel()

y_pred_group, q1, q2 = assign_tertiles(probs)

final_metrics = metrics_for_one_complication(y_true_group, y_pred_group)

print("\nFinal macro F1:")
print(final_metrics["macro_f1"])

print("\nPer class metrics:")
for cls, vals in final_metrics["per_class"].items():
    print(f"\nClass {cls}")
    print(vals)

print("\nTertile cutoffs:")
print("q1:", q1)
print("q2:", q2)

history_df


Final macro F1:
0.715620615406075

Per class metrics:

Class 0
{'TP': np.int64(766), 'FP': np.int64(234), 'FN': np.int64(150), 'precision': np.float64(0.766), 'recall': np.float64(0.8362445414847162), 'f1': np.float64(0.7995824634655532)}

Class 1
{'TP': np.int64(621), 'FP': np.int64(379), 'FN': np.int64(477), 'precision': np.float64(0.621), 'recall': np.float64(0.5655737704918032), 'f1': np.float64(0.5919923736892277)}

Class 2
{'TP': np.int64(750), 'FP': np.int64(250), 'FN': np.int64(236), 'precision': np.float64(0.75), 'recall': np.float64(0.7606490872210954), 'f1': np.float64(0.7552870090634441)}

Tertile cutoffs:
q1: 0.004940180925394584
q2: 0.01090583319247284


,round,macro_f1,q1,q2,coef_change,intercept_change
0,1,0.087319,0.157538,0.168273,0.054319,1.638424
1,2,0.103090,0.084817,0.089673,0.015945,0.708766
2,3,0.224467,0.057600,0.060457,0.024165,0.422748
3,4,0.371233,0.043520,0.045927,0.026996,0.292749
4,5,0.473864,0.034996,0.037426,0.028059,0.219873
5,6,0.540416,0.029314,0.031935,0.028399,0.173719
6,7,0.576073,0.025324,0.028092,0.028383,0.142085
7,8,0.605169,0.022334,0.025276,0.028167,0.119176
8,9,0.622291,0.020026,0.023160,0.027830,0.101901
9,10,0.633746,0.018172,0.021507,0.027414,0.088465


Bemærk i den multiple regressions model får vi 0.712 (For Nerve Dysteria). Generelt ser vi, at resultaterne er ens (+- støj). Multipel regression er mere simpel.

In [7]:
test_data.columns

Index(['Source_Client', 'Patient', 'Age', 'Sex', 'Pain', 'Swelling', 'Trismus',
       'Pericoronitis', 'Caries_Wisdom', 'Caries_Adjacent',
       'Periodontal_Status', 'Root_Development', 'Tooth_Mobility',
       'Tooth_Angulation', 'Impaction_Depth', 'Proximity_Nerve', 'Root_Count',
       'Root_Curvature', 'Bone_Density', 'Cyst', 'Diabetes', 'Osteoporosis',
       'Clotting_Disorder', 'Smoking', 'Bisphosphonates',
       'Prev_Extraction_Issue', 'Surgical_Extraction_Type', 'Score_1',
       'Score_2', 'Score_3', 'Prob_1', 'Prob_2', 'Prob_3', 'Removal_Indicated',
       'Removal_Prob', 'Risk_AlveolarOsteitis', 'Risk_AlveolarOsteitis_Prob',
       'Risk_SecondaryInfection', 'Risk_SecondaryInfection_Prob',
       'Risk_NerveDysesthesia', 'Risk_NerveDysesthesia_Prob', 'Risk_Bleeding',
       'Risk_Bleeding_Prob', 'Risk_Category_AlveolarOsteitis',
       'Risk_Category_SecondaryInfection', 'Risk_Category_NerveDysesthesia',
       'Risk_Category_Bleeding', 'Risk_Category_Composite'],
    